# Tutorial T04i: MNP Monte Carlo — Bhat (2018) Table 2 Replication.

Bhat (2018, Table 2) evaluates the accuracy and speed of five MVNCD
approximation methods (ME, OVUS, BME, TVBS, OVBS) for MNP estimation
via a Monte Carlo experiment.  This tutorial replicates the design:

  - Generate synthetic choice data from a known DGP with H+1 alternatives
  - Estimate the MNP model using each MVNCD method
  - Compare parameter recovery, log-likelihood accuracy, and computation time

The DGP follows Section 3.2 of:
    Bhat, C.R. (2018). "New matrix-based methods for the analytic evaluation
    of the multivariate cumulative normal distribution function."
    Transportation Research Part B, 109, 238-256.

What you will learn:
  - How to generate synthetic MNP data from a controlled DGP
  - How MVNCD method choice affects estimation accuracy and speed
  - How analytic vs. numerical gradients affect convergence
  - Practical guidance on method selection for different problem sizes

Prerequisites: t04a (IID model), t04d (random coefficients), t04f (control options).


In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
np.set_printoptions(precision=4, suppress=True)
import pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent / "src"))

from pybhatlib.models.mnp import MNPModel, MNPControl


# ============================================================
#  Configuration
# ============================================================
# Paper uses H in {5, 10, 15, 20} and all 5 methods.
# We use a smaller subset for reasonable runtime.
H_VALUES = [5, 10]                 # H+1 = 6, 11 alternatives
N_OBS = 3000                       # same as paper
METHODS = ["me", "ovus", "tvbs"]   # subset for speed
REFERENCE_METHOD = "tvbs"          # highest-accuracy analytic method
MAXITER = 200
SEED = 42


## Step 1: Data Generating Process (Bhat 2018, Sec. 3.2)


In [ ]:
print("""
  The Monte Carlo DGP generates choice data for H+1 alternatives
  (indexed h = 0, 1, ..., H) and N = 3000 individuals (indexed q).

  Each individual q choosing among H+1 alternatives has utility:

      U_qh = b1 * x1_qh + b2 * x2_qh + gamma_q * z_qh + xi_qh

  where:
    x1_qh ~ Uniform(0, 1)         continuous covariate
    x2_qh ~ Bernoulli(0.5)        dummy variable (see below)
    z_qh  ~ Normal(0, 1)          random-coefficient covariate
    gamma_q ~ Normal(c, tau^2)    individual-specific random coefficient
    xi_qh ~ Normal(0, 0.5)        IID error (variance = 0.5)

  True parameter values:
    b1    = 1.0       coefficient on x1 (generic, all alternatives)
    b2    = 0.75      coefficient on x2 (active for second half only)
    c     = -0.5      mean of random coefficient on z
    tau^2 = 1.0       variance of random coefficient on z

  Alternative-specific structure of x2:
    b2 = 0   for h = 0, 1, ..., floor(H/2)     (first half)
    b2 = 0.75 for h = floor(H/2)+1, ..., H     (second half)

  This is implemented by setting x2_qh = 0 for the first half of
  alternatives, so the coefficient b2 is only identified from the
  second half.

  The individual chooses the alternative with highest utility:
      y_q = argmax_h U_qh
""")


## Step 2: Synthetic Data Generation


In [ ]:
def generate_mnp_data(H, N, seed=42):
    """Generate synthetic MNP data following Bhat (2018) Section 3.2.

    Parameters
    ----------
    H : int
        Number of alternatives minus 1 (so H+1 total alternatives).
    N : int
        Number of observations.
    seed : int
        Random seed.

    Returns
    -------
    df : pd.DataFrame
        DataFrame with all variable columns and choice indicators.
    alternatives : list of str
        Names of choice indicator columns.
    spec : dict
        Variable specification dict for MNPModel.
    """
    rng = np.random.default_rng(seed)
    n_alts = H + 1

    # Individual-level random coefficient: gamma_q ~ N(c=-0.5, tau=1)
    gamma = rng.normal(-0.5, 1.0, size=N)

    data = {}

    # Generate variables for each alternative
    for h in range(n_alts):
        # X1: continuous U[0,1] — generic across all alternatives
        data[f"X1_{h}"] = rng.uniform(0, 1, size=N)

        # X2: dummy variable — only active for second half of alternatives
        # For first half (h <= H//2), column is zero so b2 * 0 = 0
        if h <= H // 2:
            data[f"X2_{h}"] = np.zeros(N)
        else:
            data[f"X2_{h}"] = rng.binomial(1, 0.5, size=N).astype(float)

        # Z: random-coefficient covariate, continuous N(0,1)
        data[f"Z_{h}"] = rng.normal(0, 1, size=N)

    # Compute utilities and determine choices
    utilities = np.zeros((N, n_alts))
    for h in range(n_alts):
        b1, b2 = 1.0, 0.75
        utilities[:, h] = (
            b1 * data[f"X1_{h}"]
            + (b2 * data[f"X2_{h}"] if h > H // 2 else 0.0)
            + gamma * data[f"Z_{h}"]
            + rng.normal(0, np.sqrt(0.5), size=N)  # IID error, var=0.5
        )

    choices = np.argmax(utilities, axis=1)

    # Choice indicator columns
    alternatives = [f"Alt{h}_ch" for h in range(n_alts)]
    for h in range(n_alts):
        data[alternatives[h]] = (choices == h).astype(float)

    # Constant columns needed by spec
    data["uno"] = np.ones(N)
    data["sero"] = np.zeros(N)

    df = pd.DataFrame(data)

    # Build variable specification
    spec = {}

    # X1: generic variable across all alternatives
    spec["X1"] = {alternatives[h]: f"X1_{h}" for h in range(n_alts)}

    # X2: only active for second half; first half maps to "sero"
    spec["X2"] = {}
    for h in range(n_alts):
        if h > H // 2:
            spec["X2"][alternatives[h]] = f"X2_{h}"
        else:
            spec["X2"][alternatives[h]] = "sero"

    # Z: random-coefficient variable (generic across all alternatives)
    spec["Z"] = {alternatives[h]: f"Z_{h}" for h in range(n_alts)}

    return df, alternatives, spec


# Generate and inspect data for each H
datasets = {}
for H in H_VALUES:
    df, alt_cols, spec = generate_mnp_data(H, N_OBS, seed=SEED)
    datasets[H] = (df, alt_cols, spec)

    n_alts = H + 1
    choice_counts = df[alt_cols].sum(axis=0).values.astype(int)

    print(f"\n  H = {H} ({n_alts} alternatives), N = {N_OBS}")
    print(f"  DataFrame shape: {df.shape}")
    print(f"  Variables per alternative: X1, X2, Z")
    print(f"  Choice distribution:")
    for h in range(n_alts):
        pct = choice_counts[h] / N_OBS * 100
        print(f"    Alt {h}: {choice_counts[h]:>5d} ({pct:5.1f}%)")


## Step 3: Variable Specification


In [ ]:
# Show spec for smallest H as an example
H_example = H_VALUES[0]
_, alt_example, spec_example = datasets[H_example]

print(f"""
  For H = {H_example} ({H_example + 1} alternatives), the spec dict maps
  variable names to alternative-specific data columns.

  Key design pattern for alternative-specific availability:
    - X2 has a nonzero coefficient only for the second half of alternatives.
    - For the first half, spec maps to "sero" (a column of zeros),
      so b2 * sero = 0 regardless of the estimated b2 value.
    - This is cleaner than creating separate variables per alternative subset.
""")

print("  spec['X1'] (generic across all alternatives):")
for alt, col in spec_example["X1"].items():
    print(f"    {alt} -> {col}")

print(f"\n  spec['X2'] (active only for h > {H_example // 2}):")
for alt, col in spec_example["X2"].items():
    marker = "  <-- sero (inactive)" if col == "sero" else ""
    print(f"    {alt} -> {col}{marker}")

print("\n  spec['Z'] (random coefficient, generic):")
for alt, col in spec_example["Z"].items():
    print(f"    {alt} -> {col}")


## Step 4: Reference Estimation (TVBS)


In [ ]:
print(f"""
  We first estimate the model using TVBS (Tri-Variate Bivariate Screening),
  the highest-accuracy analytic MVNCD method in our benchmarks
  (MAPE = 0.25% at K=5, see t03b_bhat2018_table1.py).

  TVBS estimates serve as the reference against which we compare other
  methods.  The comparison uses:
    - Parameter MAPE: Mean Absolute Percentage Error of estimated parameters
    - LL APE: Absolute Percentage Error of total log-likelihood
    - Computation time in seconds
""")

reference_results = {}
for H in H_VALUES:
    df, alt_cols, spec = datasets[H]

    print(f"\n  Estimating H = {H} ({H + 1} alternatives) with TVBS...")

    ctrl = MNPControl(
        iid=True,
        method="tvbs",
        maxiter=MAXITER,
        verbose=0,
        seed=SEED,
        analytic_grad=False,  # TVBS uses numerical gradient
    )

    t0 = time.perf_counter()
    model = MNPModel(
        data=df,
        alternatives=alt_cols,
        spec=spec,
        mix=True,
        ranvars=["Z"],
        control=ctrl,
    )
    results = model.fit()
    elapsed = time.perf_counter() - t0

    reference_results[H] = results

    print(f"    LL = {results.ll_total:.3f}")
    print(f"    Converged: {results.converged} ({results.n_iterations} iterations)")
    print(f"    Time: {elapsed:.1f}s")
    print(f"    Parameters: ", end="")
    for name, val in zip(results.param_names, results.b):
        print(f"{name}={val:.4f}  ", end="")
    print()


## Step 5: MVNCD Method Comparison


In [ ]:
print(f"""
  We now estimate the same model using each MVNCD method and compare
  against the TVBS reference.

  Methods tested: {', '.join(m.upper() for m in METHODS)}

  For the ME (Matrix Exponential) method, we run two variants:
    - ME (analytic): analytic_grad=True  — fastest, gradient computed in closed form
    - ME (numeric):  analytic_grad=False — numerical gradient via finite differences

  All other methods use numerical gradients (analytic gradients are
  currently implemented only for the ME method).
""")

# Store all results: {H: {method_label: {params, ll, time, converged, n_iter}}}
all_results = {}

for H in H_VALUES:
    df, alt_cols, spec = datasets[H]
    all_results[H] = {}

    ref = reference_results[H]

    # Store reference in results dict
    all_results[H]["TVBS (ref)"] = {
        "params": ref.b.copy(),
        "ll": ref.ll_total,
        "time": ref.convergence_time * 60.0,  # convert minutes to seconds
        "converged": ref.converged,
        "n_iter": ref.n_iterations,
    }

    for method in METHODS:
        if method == REFERENCE_METHOD:
            # Already estimated as reference; skip re-estimation
            continue

        if method == "me":
            # Run ME twice: with and without analytic gradient
            for use_analytic, label in [(True, "ME (analytic)"), (False, "ME (numeric)")]:
                ctrl = MNPControl(
                    iid=True,
                    method="me",
                    maxiter=MAXITER,
                    verbose=0,
                    seed=SEED,
                    analytic_grad=use_analytic,
                )

                t0 = time.perf_counter()
                model = MNPModel(
                    data=df,
                    alternatives=alt_cols,
                    spec=spec,
                    mix=True,
                    ranvars=["Z"],
                    control=ctrl,
                )
                results = model.fit()
                elapsed = time.perf_counter() - t0

                all_results[H][label] = {
                    "params": results.b.copy(),
                    "ll": results.ll_total,
                    "time": elapsed,
                    "converged": results.converged,
                    "n_iter": results.n_iterations,
                }

                print(f"  H={H}, {label}: LL={results.ll_total:.3f}, "
                      f"{results.n_iterations} iter, {elapsed:.1f}s")
        else:
            label = method.upper()
            ctrl = MNPControl(
                iid=True,
                method=method,
                maxiter=MAXITER,
                verbose=0,
                seed=SEED,
                analytic_grad=False,
            )

            t0 = time.perf_counter()
            model = MNPModel(
                data=df,
                alternatives=alt_cols,
                spec=spec,
                mix=True,
                ranvars=["Z"],
                control=ctrl,
            )
            results = model.fit()
            elapsed = time.perf_counter() - t0

            all_results[H][label] = {
                "params": results.b.copy(),
                "ll": results.ll_total,
                "time": elapsed,
                "converged": results.converged,
                "n_iter": results.n_iterations,
            }

            print(f"  H={H}, {label}: LL={results.ll_total:.3f}, "
                  f"{results.n_iterations} iter, {elapsed:.1f}s")


## Step 6: Results Tables


In [ ]:
print("""
  Metrics:
    Param-MAPE = mean |theta_method - theta_ref| / |theta_ref| * 100
                 averaged over all estimated parameters
    LL-APE     = |LL_method - LL_ref| / |LL_ref| * 100
    Time       = wall-clock estimation time in seconds

  The reference method (TVBS) has dashes for MAPE and LL-APE since
  it is the comparison baseline.
""")

for H in H_VALUES:
    ref_entry = all_results[H]["TVBS (ref)"]
    ref_params = ref_entry["params"]
    ref_ll = ref_entry["ll"]

    n_alts = H + 1
    print(f"\n  H = {H} ({n_alts} alternatives)")
    print(f"  Reference LL (TVBS): {ref_ll:.3f}")
    print(f"  Reference params: {ref_params}")
    print()

    header = f"  {'Method':<16s} {'Param-MAPE':>11s} {'LL-APE':>9s} {'Time(s)':>9s} {'Iters':>7s} {'Converged':>10s}"
    print(header)
    sep_line = "  " + "-" * 65
    print(sep_line)

    # Determine display order
    display_order = []
    for label in ["ME (analytic)", "ME (numeric)", "OVUS", "TVBS (ref)"]:
        if label in all_results[H]:
            display_order.append(label)
    # Add any remaining methods not in the predefined order
    for label in all_results[H]:
        if label not in display_order:
            display_order.append(label)

    for label in display_order:
        entry = all_results[H][label]

        if label == "TVBS (ref)":
            param_mape_str = "---"
            ll_ape_str = "---"
        else:
            # Parameter MAPE: skip near-zero reference params to avoid division issues
            mapes = []
            for p_est, p_ref in zip(entry["params"], ref_params):
                if abs(p_ref) > 1e-6:
                    mapes.append(abs(p_est - p_ref) / abs(p_ref) * 100)
            param_mape = np.mean(mapes) if mapes else 0.0
            param_mape_str = f"{param_mape:.2f}%"

            # LL APE
            ll_ape = abs(entry["ll"] - ref_ll) / abs(ref_ll) * 100
            ll_ape_str = f"{ll_ape:.4f}%"

        conv_str = "Yes" if entry["converged"] else "No"
        print(f"  {label:<16s} {param_mape_str:>11s} {ll_ape_str:>9s} "
              f"{entry['time']:>8.1f}s {entry['n_iter']:>7d} {conv_str:>10s}")

    print(sep_line)


## Step 7: Analytic vs. Numerical Gradient (ME Method)


In [ ]:
print("""
  The ME (Matrix Exponential) method is the only MVNCD method for which
  pybhatlib has an analytic gradient implementation.  This enables the
  optimizer to compute exact gradients instead of using finite-difference
  approximation.

  Impact on estimation:
    - Analytic gradient is faster per iteration (no 2*n_params extra LL evals)
    - Analytic gradient provides exact direction, potentially fewer iterations
    - Numerical gradient may differ slightly due to finite-difference error

  We compare the two variants side by side.
""")

for H in H_VALUES:
    n_alts = H + 1
    print(f"\n  H = {H} ({n_alts} alternatives):")

    analytic_entry = all_results[H].get("ME (analytic)")
    numeric_entry = all_results[H].get("ME (numeric)")

    if analytic_entry is None or numeric_entry is None:
        print("    (ME results not available for this H)")
        continue

    print(f"    {'':>20s} {'Analytic':>12s} {'Numerical':>12s}")
    print(f"    {'':>20s} {'-'*12:>12s} {'-'*12:>12s}")
    print(f"    {'Log-likelihood':>20s} {analytic_entry['ll']:>12.3f} {numeric_entry['ll']:>12.3f}")
    print(f"    {'Iterations':>20s} {analytic_entry['n_iter']:>12d} {numeric_entry['n_iter']:>12d}")
    print(f"    {'Time (seconds)':>20s} {analytic_entry['time']:>12.1f} {numeric_entry['time']:>12.1f}")
    print(f"    {'Converged':>20s} {'Yes' if analytic_entry['converged'] else 'No':>12s} "
          f"{'Yes' if numeric_entry['converged'] else 'No':>12s}")

    if analytic_entry["time"] > 0 and numeric_entry["time"] > 0:
        speedup = numeric_entry["time"] / analytic_entry["time"]
        print(f"    {'Speedup factor':>20s} {speedup:>12.1f}x {'':>12s}")

    # Parameter comparison
    ll_diff = abs(analytic_entry["ll"] - numeric_entry["ll"])
    param_diff = np.max(np.abs(analytic_entry["params"] - numeric_entry["params"]))
    print(f"\n    LL difference (analytic - numeric): {analytic_entry['ll'] - numeric_entry['ll']:.4f}")
    print(f"    Max parameter difference:           {param_diff:.6f}")

    if ll_diff < 0.1:
        print("    Both variants converge to effectively the same solution.")
    else:
        print("    Note: Differences may arise from different optimization paths.")


## Step 8: Interpretation and Practical Guidance


In [ ]:
print("""
  Summary of findings from this Monte Carlo experiment:

  1. ACCURACY:
     All tested MVNCD methods converge to very similar parameter estimates
     and log-likelihoods.  The parameter MAPE relative to the TVBS reference
     is typically below 1-2%, confirming that method choice has minimal
     impact on final estimates for moderate problem sizes (H <= 10).

  2. SPEED:
     - ME (analytic grad) is the fastest method, benefiting from exact
       gradients that avoid 2*n_params extra likelihood evaluations per
       iteration.
     - ME (numeric grad) is slower because finite differences require
       many extra function evaluations.
     - OVUS and TVBS computation time grows with H because the MVNCD
       approximation complexity increases with dimensionality.

  3. ANALYTIC GRADIENT IMPACT:
     The analytic gradient for the ME method provides a substantial speedup
     (often 2-5x faster) with no loss in accuracy.  For large-scale
     applications, this is the recommended configuration.

  4. WHEN TO USE EACH METHOD:
     - ME (analytic):  Default choice for IID models.  Fastest estimation,
                       good accuracy.  Use analytic_grad=True.
     - OVUS:           Good accuracy-speed balance for flexible covariance.
                       Default in pybhatlib.
     - TVBS:           Highest accuracy among analytic methods.  Use when
                       accuracy is paramount (e.g., final reported results).
     - BME / OVBS:     Similar accuracy to OVUS; useful for robustness checks.

  5. SCALE NORMALIZATION:
     The true DGP has IID error variance = 0.5, but the MNP estimator
     normalizes the differenced error covariance to ones+I (diagonal = 2,
     off-diagonal = 1, i.e., unit variance per undifferenced error).
     Estimated coefficients are therefore scaled relative to true values.
     Since all methods use the same normalization, cross-method comparison
     (as done here) is valid without scale adjustment.

  6. LIMITATIONS OF THIS EXERCISE:
     - We use a single dataset per H value (not the full 50+ replications
       of a proper Monte Carlo study as in the paper).
     - Per-observation log-likelihood comparison is not included (would
       require library modification to expose individual contributions).
     - Larger H values (15, 20) are omitted for runtime but can be enabled
       by modifying H_VALUES at the top of this script.
     - BME, OVBS, and SSJ methods are not tested by default but can be
       added to the METHODS list.
""")

# Print configuration reminder
print("  Configuration used:")
print(f"    H values:         {H_VALUES}")
print(f"    N observations:   {N_OBS}")
print(f"    Methods tested:   {', '.join(m.upper() for m in METHODS)}")
print(f"    Reference method: {REFERENCE_METHOD.upper()}")
print(f"    Max iterations:   {MAXITER}")
print(f"    Random seed:      {SEED}")
print()
print("  To run the full Table 2 experiment, modify the configuration at")
print("  the top of this script:")
print("    H_VALUES = [5, 10, 15, 20]")
print('    METHODS = ["me", "ovus", "bme", "tvbs", "ovbs"]')
print()
print("  Next: explore other pybhatlib tutorials for flexible covariance,")
print("  mixture-of-normals, and forecasting applications.")
